---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-32: LangChain <b>Memory</b> Component</h1>

# Learning agenda of this notebook  
1. Recap: Core Components of LangChain
2. Stateless Nature of AI Models
    - Example 1: A Q/A Bot Without Chat History
    - Example 2: A Conversational ChatBot by Mannually managing Short-Term / Working memory
3. Stateful Chatbot using LangChain Memory Component    
    - Example 3: A Conversational ChatBot using Langchain's `ChatMessageHistory`
4. Characteristics and Limitations of Short Term Memory (Context Window)
5. Handling Overflow of Context Window using LangChain Memory Component
    - Example 4: A Conversational ChatBot using a Sliding Window and retaining recent N messages
    - Example 5: A Conversational ChatBot that summarizes conversation history
    - Example 6: A Conversational ChatBot that manages multiple independent conversation sessions

# <span style='background :lightgreen' >1. Recap: Core Components of [LangChain](https://github.com/langchain-ai/langchain)</span>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain is an open source framework that provides us with a set of tools and abstractions that make it easier for us to create complex LLM-powered applications.</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Langchain components can be chained together to create sophisticated AI applications like chatbots, question-answering systems, and intelligent agents.</h3>


| Component | Description | Key Benefit |
|-----------|--------------|-------------|
| **Models** | LangChain's "universal remote control" for AI—one interface to rule them all, freeing developers from vendor-specific complexity and enabling true AI provider independence. | Write once, use with any AI model |
| **Prompts** | Reusable, parameterized templates that standardize how you communicate with AI models. | Consistent messaging with easy updates | 
| **Memory** | LangChain Memory enables AI models to maintain context across multiple interactions in a conversation. Without memory, each interaction is independent - the model has no knowledge of previous exchanges. | Natural, flowing conversations |
| **Chains** | LangChain’s “AI Assembly Lines” — they connect multiple models, prompts, or tools together into a logical flow of reasoning, automating complex multi-step tasks with simplicity and consistency. | Transform complex tasks into simple sequences | 
| **Indexes** | Indexes are LangChain's smart knowledge management systems that convert your raw documents (PDFs, websites, databases) into searchable libraries where AI models can quickly find and retrieve only the most relevant information w/o exceeding context limits. | Fast retrieval, reduced API costs | 
| **Agents** | LangChain Agents are the “decision-making brains” of your AI system — they enable models to reason, plan, and dynamically decide which tools to use and what actions to take in order to achieve a user’s goal.| Autonomous problem-solving without manual programming |

# <span style='background :lightgreen' >2. Stateless Nature of AI Models</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">LLMs used through APIs operate in a stateless manner, i.e., the model does not remember anything about previous messages, unless the developer explicitly manages it.</h3>


<img align="right" width="900"  src="../images/memory11.png"  > 


## What Does “Stateless” Mean?
- Each request is processed independently, with no stored memory from previous interactions. For LLMs the API treats every call as a fresh, isolated request. The model does not persist:
    - chat history
    - user identity
    - previous answers
    - conversation context

<br><br>

## Why Are LLM APIs Stateless?
- **Scalability:** Stateless systems are easier to scale horizontally across thousands of servers.
- **Security & Privacy:** Avoids storing personal or sensitive conversation history on servers.
- **Model Behavior Predictability:** Developers have full control of what context the model sees.
- **Simplicity of API Design:** Each request is self-contained → fewer failure points.


## Example 1: A Q/A Bot **Without** Chat History

In [1]:
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

while True:
    user_input = input('You: ')
    if user_input == 'exit':
        print("👋 Goodbye!")
        break
    result = llama_model.invoke(user_input)
    print("AI: ",result.content)

You:  hello i am arif


AI:  Hello Arif, it's nice to meet you. Is there something I can help you with or would you like to chat?


You:  can you tell me my name


AI:  I don't have the ability to know your name. I'm a large language model, I don't have personal interactions or retain information about individual users. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation.


You:  exit


👋 Goodbye!


## Example 2: A Conversational ChatBot manually managing a Chat History List
- **Short-term memory** in LLM applications is the information it has about recent interactions, typically the ongoing conversations with the user or the behavior of the LLM. In LLM applications, this manifests as the conversation history that persists across API calls and is continuously provided to the model as context.
- The most common way to implement short-term memory is to store the entire conversation history and send it to the model on every request.
    - **Using Chat Completion APIs:**
        - Maintain a list of messages containing all system, user, and assistant messages.
        - Send this entire list in every request to the API.
        - This approach is simple but rapidly consumes the token budget as the conversation grows.
    - **Using Responses API:**
        - Instead of manually managing the message list, the API allows referencing the previous response using the parameter `previous_response_id`.
        - The server manages conversation history internally, reducing the need to resend all messages each time.
    - **Using LangChain:**
        - In LangChain, a common beginner-level approach is to maintain a Python list called `chat_history`.
        - Each interaction is appended to this list using message objects:
            - `SystemMessage()` → defines system instructions
            - `HumanMessage()` → represents the user input
            - `AIMessage()` → represents the model response
        - The entire chat_history list is sent to the model on every invocation, as shown in the example code:

In [2]:
##################      Mannually managage a list of SystemMessage, HummanMessage and AI Message         #######################
import os                                           # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                      # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI             # LangChain wrapper for OpenAI models (e.g. GPT-4o)      
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


chat_history = [SystemMessage(content='You are a helpful AI assistant')] # Initializes a list with the system message

while True:
    user_input = input('You: ')
    if user_input == 'exit':
        print("👋 Goodbye!")
        break
    chat_history.append(HumanMessage(content=user_input))    # Add user message to chat_history
    result = llama_model.invoke(chat_history)               # Send entire chat history to model
    chat_history.append(AIMessage(content=result.content))  # Add AI response to chat_history
    print("AI:", result.content)

# Display the final chat history (for debugging)
print("\n🧠 Final Chat History (for understanding message flow):")
for msg in chat_history:
    print(f"{msg.type.upper()}: {msg.content}")

You:  hello i am arif


AI: Hello Arif, it's nice to meet you. Is there something I can help you with or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.


You:  can you tell me my name


AI: Your name is Arif.


You:  exit


👋 Goodbye!

🧠 Final Chat History (for understanding message flow):
SYSTEM: You are a helpful AI assistant
HUMAN: hello i am arif
AI: Hello Arif, it's nice to meet you. Is there something I can help you with or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
HUMAN: can you tell me my name
AI: Your name is Arif.


# <span style='background :lightgreen' >3. Stateful Chatbot using LangChain Memory Component</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain Memory enables AI models to maintain context across multiple interactions in a conversation </h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">LangChain provides several built-in memory modules that automatically manage conversation history and help prevent context window overflow </h3>

## Example 3: A Conversational ChatBot using Langchain's `ChatMessageHistory`
- Instead of managing a raw list, LangChain provides a memory object that internally stores messages.
```python
from langchain_community.chat_message_histories import ChatMessageHistory
memory = ChatMessageHistory()                                                 # Initialize ChatMessageHistory (replaces ConversationBufferMemory in v1.x)
memory.add_message(SystemMessage(content='You are a helpful AI assistant'))   # Add system message to memory
memory.add_user_message(user_input)                                           # Add user message to memory
result = llama_model.invoke(memory.messages)                                  # Get all messages from memory and send to model
memory.add_ai_message(result.content)                                         # Add AI response to memory
```

In [1]:
##################  Use LangChain's ChatMessageHistory object and use add_message(), add_user_message() and add_ai_message()     #######################
import os                                                                  # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                                             # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI                                    # LangChain wrapper for OpenAI models (e.g. GPT-4o)     
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles
from langchain_community.chat_message_histories import ChatMessageHistory  # LangChain utility class that stores and manages conversation history (system, user, and AI messages) for chat applications

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")



# Initialize ChatMessageHistory (replaces ConversationBufferMemory in v1.x)
memory = ChatMessageHistory()

# Add system message to memory
memory.add_message(SystemMessage(content='You are a helpful AI assistant'))

print("Chat started! Type 'exit' to quit.\n")

while True:
    user_input = input('You: ')
    if user_input.lower() == 'exit':
        print("👋 Goodbye!")
        break
    memory.add_user_message(user_input)             # Add user message to memory
    result = llama_model.invoke(memory.messages)    # Get all messages from memory and send to model
    memory.add_ai_message(result.content)          # Add AI response to memory
    print("AI:", result.content)
    print()

# Optional: Display full conversation history from memory
print("\n🧠 Full Conversation History from Memory:")
for msg in memory.messages:
    if isinstance(msg, SystemMessage):
        print(f"System: {msg.content}")
    elif isinstance(msg, HumanMessage):
        print(f"Human: {msg.content}")
    elif isinstance(msg, AIMessage):
        print(f"AI: {msg.content}")

Chat started! Type 'exit' to quit.



You:  hello, i am Arif


AI: Hello Arif, it's nice to meet you. Is there something I can help you with or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss. How's your day going so far?



You:  Can you tell me my name?


AI: Your name is Arif. You told me that when you introduced yourself.



You:  exit


👋 Goodbye!

🧠 Full Conversation History from Memory:
System: You are a helpful AI assistant
Human: hello, i am Arif
AI: Hello Arif, it's nice to meet you. Is there something I can help you with or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss. How's your day going so far?
Human: Can you tell me my name?
AI: Your name is Arif. You told me that when you introduced yourself.


# <span style='background :lightgreen' >4. Characteristics and Limitations of Short Term Memory (Context Window)</span>
## a. Key Characteristics of Short-Term Memory
- **Session-Based Persistence:** Memory exists only during the current session and must be re-sent with every interaction. The model itself does not permanently remember past conversations.
- **Context Window Constraints:** The conversation history is included inside the prompt and therefore counts toward the model’s token limit.
- **Dynamic Memory Management:** As conversations grow, the application must actively manage which parts of the history to keep, compress, or discard.


## b. Limitations of Short-Term Memory
- **Limited Context Window:** The model can only process a finite number of tokens, so very long conversations cannot be preserved entirely.
- **Session Loss:** If the program restarts, the server crashes, or the user opens a new chat session, the conversation history may be lost unless it has been stored externally.
- **Conversation Isolation:** Each new conversation thread starts with no knowledge of previous chats unless long-term storage mechanisms are used.



## c. Overflow of AI Model's Context Window

<div style="flex: 1; text-align: center;">
    <img src="../images/context-window.png" style="max-width: 1000px; height: 1000px;">
</div>


## d. Techniques to Keep Conversation  History within the Model's Context Window
- **Sliding Window (Recent Message Retention):**
    - Retains only the most recent N messages, creating a moving window through the conversation.
    - It can be implemented using a fixed-size buffer (e.g., FIFO queue) that automatically discards the oldest message when a new one arrives.
    - It is simple and efficient and maintains the most recent conversational context.
    - Its limitation is that important details from the beginning of the conversation may disappear.
- **Token-Based Truncation:** Manages memory based on token consumption rather than message count, maintaining a "token budget." It can be implemented by tracking cumulative token count of messages; when approaching the limit, remove oldest messages until within budget. This technique may discard important early information and requires token counting logic.
    - Instead of limiting by message count, this method limits by total token usage.
    - It can be implemented by tracking the token count of the entire conversation history. When the token limit is approached, remove the oldest messages until the prompt fits within the allowed budget.
    - It provides more precise control over the context window and better utilization of available tokens.
    - However, it requires token counting logic and important details from the beginning of the conversation may disappear.
-  **Conversation Summarization:** Generates compressed summaries of older conversation segments, similar to meeting notes. It can be implemented by periodically invoking the LLM to create concise summaries of older exchanges; replace original messages with summaries to save tokens. It requires additional API calls to LLM for summary generation and still the risk of losing granular details exist.
    - This technique compresses earlier messages into short summaries, similar to meeting notes.
    - Periodically call the LLM to summarize older parts of the conversation and replace the original messages with the generated summary
    - It preserves the meaning of earlier discussions and allows much longer conversations
    - However, it requires additional LLM API calls and fine-grained details may be lost during summarization
- **Context Switching:** Dynamically adjusts focus based on detected topic changes, prioritizing relevant context. It can be implemented using topic modeling or classification to detect conversation shifts. The programmer needs to maintain separate context buffers for different topics or reset context when topics change. For example: When conversation shifts from travel planning to cooking recipes, the system deprioritizes flight details and focuses on cooking recipes.
    - In some applications, conversations naturally shift between topics. Context switching focuses on retaining only the information relevant to the current topic.
    - Use topic detection or classification and maintain separate context buffers for different topics. Reset or deprioritize old contexts when the topic changes
    - For example if a conversation shifts from travel planning to cooking recipes, the system may discard travel-related details and prioritize cooking-related context.

# <span style='background :lightgreen' >5. Handling Overflow of Context Window using LangChain Memory Component</span>
## Example 4: A Conversational ChatBot using a Sliding Window and retaining recent N messages
- **Can be implemented manually using Langchain's `ChatMessageHistory` OR using Langchain's  `ConversationBufferWindowMemory`**

In [7]:
import os                                                                  # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                                             # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI                                    # LangChain wrapper for OpenAI models (e.g. GPT-4o)     
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles
from langchain_community.chat_message_histories import ChatMessageHistory  # LangChain utility class that stores and manages conversation history (system, user, and AI messages) for chat applications

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")



memory = ChatMessageHistory()                                                # Initialize ChatMessageHistory (replaces ConversationBufferMemory in v1.x)
memory.add_message(SystemMessage(content='You are a helpful AI assistant who answers in a brief fashion'))  # Add system message to memory

print("Chat started! Type 'exit' to quit.\n")

WINDOW_SIZE = 4         # Keep system message and only last 4 messages (2 user messages + 2 AI responses)


while True:
    user_input = input('You: ')
    if user_input.lower() == 'exit':
        print("👋 Goodbye!")
        break
    memory.add_user_message(user_input)            # Add user message to memory
    result = llama_model.invoke(memory.messages)   # Get all messages from memory and send to model
    memory.add_ai_message(result.content)          # Add AI response to memory
    
    # Keep only system message + last 2 conversations (4 messages)
    if len(memory.messages) > WINDOW_SIZE:                                        # +1 for system message
        memory.messages = [memory.messages[0]] + memory.messages[-WINDOW_SIZE:]   # Keep last 4 messages along with the system message
    
    print("AI:", result.content)
    print()

# Optional: Display conversation history inside the Sliding window
print("\n🧠 Conversation History from Sliding Window:")
for msg in memory.messages:
    if isinstance(msg, SystemMessage):
        print(f"System: {msg.content}")
    elif isinstance(msg, HumanMessage):
        print(f"Human: {msg.content}")
    elif isinstance(msg, AIMessage):
        print(f"AI: {msg.content}")

Chat started! Type 'exit' to quit.



You:  hello i am arif


AI: Hello Arif, how can I help you?



You:  can you tell me my name


AI: Your name is Arif.



...

AI: Guido van Rossum created Python.



You:  can you tell me my name


AI: I don't know your name.



You:  exit


👋 Goodbye!

🧠 Full Conversation History from Memory:
System: You are a helpful AI assistant who answers in a brief fashion
Human: who invented it
AI: Guido van Rossum created Python.
Human: can you tell me my name
AI: I don't know your name.


## Example 5: A Conversational ChatBot that summarizes conversation history
- Can be implemented manually using Langchain's `ChatMessageHistory` OR using Langchain's `ConversationSummaryMemory`
- **What This Does:**
    - Tracks exchange count - Counts how many back-and-forth exchanges happened
    - Summarizes after N exchanges - After 3 exchanges, ask LLM to create a summary
    - Clears old messages - Replaces detailed history with compact summary
    - Updates system message - Includes the summary so context isn't lost
    - Reset counter

In [2]:
import os                                                                  # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                                             # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI                                    # LangChain wrapper for OpenAI models (e.g. GPT-4o)     
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles
from langchain_community.chat_message_histories import ChatMessageHistory  # LangChain utility class that stores and manages conversation history (system, user, and AI messages) for chat applications

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")



memory = ChatMessageHistory()                                                # Initialize ChatMessageHistory (replaces ConversationBufferMemory in v1.x)
memory.add_message(SystemMessage(content='You are a helpful AI assistant who answers in a brief fashion'))  # Add system message to memory

conversation_summary = "No previous conversation."
SUMMARIZE_AFTER = 3  # Summarize after this many exchanges
exchange_count = 0

print("Chat started! Type 'exit' to quit.\n")

while True:
    user_input = input('You: ')
    if user_input.lower() == 'exit':
        print("👋 Goodbye!")
        break
    memory.add_user_message(user_input)     # Add user message to memory
    result = llama_model.invoke(memory.messages) # Get all messages from memory and send to model
    memory.add_ai_message(result.content)  # Add AI response to memory
    print("AI:", result.content)
    
    exchange_count += 1
    
    # Summarize conversation after every N exchanges
    if exchange_count >= SUMMARIZE_AFTER:
        print("\n[Creating summary...]\n")
        # Build conversation text from memory
        conversation_text = ""
        for msg in memory.messages[1:]:                                 # Skip system message
            role = "User" if isinstance(msg, HumanMessage) else "AI"
            conversation_text += f"{role}: {msg.content}\n"
        
        # Generate summary via model
        summary_prompt = f"Summarize this conversation concisely:\n\n{conversation_text}\nSummary:"
        summary_result = llama_model.invoke([HumanMessage(content=summary_prompt)])
        conversation_summary = summary_result.content
        
        # Clear memory but keep system message with summary
        memory.clear()
        memory.add_message(SystemMessage(content=f"You are a helpful AI assistant who answers in a brief fashion. Previous conversation summary: {conversation_summary}"))
        
        exchange_count = 0
        print(f"[Summary: {conversation_summary}]\n")

# Show final summary
print("\n--- 🧠 Final Conversation Summary ---")
print(conversation_summary)


Chat started! Type 'exit' to quit.



You:  hello, i am Arif


AI: Hello Arif, how can I help you today?


You:  My dogs name is zoro


AI: That's a cool name. Is Zoro a playful pup?


...

AI: The color of the sky is blue.


You:  Can you tell which course i am teaching in the current semester and the name of my dog?


AI: You, Professor Arif, are teaching a course on Generative and Agentic AI, and your dog's name is Zoro.


You:  exit


👋 Goodbye!

--- 🧠 Final Conversation Summary ---
Arif, a professor, introduced himself and his dog Zoro, and discussed his current semester course on Generative and Agentic AI.


## Example 6: A Conversational ChatBot that manages multiple independent conversation sessions

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The program implements a multi-session conversational chatbot using LangChain and Groq-hosted LLaMA model, where each session maintains its own independent conversation history.</h3>

- **Memory Management (ChatMessageHistory):**
    - Each conversation session has its own ChatMessageHistory object.
    - This stores messages as structured objects (SystemMessage, HumanMessage, AIMessage) instead of plain text.
    - Provides easy methods like .add_user_message() and .add_ai_message() to update chat history.
- **Session-Based Conversation Control:**
    - A dictionary named sessions maps session names to their chat histories.
    - The user starts in a default session (session_1) but can:
        - Switch to another session (switch <name>)
        - Create a new session if it doesn’t exist
        - List all active sessions using the list command
- **Conversational Flow:**
    - For each user message:
        - Adds the message to the session’s chat history.
        - Sends the entire message history to the model via llama_model.invoke(memory.messages).
        - Appends the model’s response back into the session memory.
- **Multi-Session Context Preservation:**
    - Switching sessions preserves past histories — each session continues from its previous context.
    - Sessions are fully isolated: the AI remembers context only within the current session.
- **Summary and Exit:**
    - On exit, the program displays a summary of all sessions, printing each session’s conversation history (system, user, and AI messages).

In [9]:
import os                                                                  # Access environment variables (e.g. os.getenv("API_KEY"))
from dotenv import load_dotenv                                             # Load API keys from a .env file into environment variables         
from langchain_openai import ChatOpenAI                                    # LangChain wrapper for OpenAI models (e.g. GPT-4o)     
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage # Classes to represent chat roles
from langchain_community.chat_message_histories import ChatMessageHistory  # LangChain utility class that stores and manages conversation history (system, user, and AI messages) for chat applications

load_dotenv('../keys/.env', override=True)          # Load variables from .env file into system environment
groq_api_key = os.getenv('GROQ_API_KEY')            # Retrieve Groq API key (used to access LLaMA model)
llama_model = ChatOpenAI(model="llama-3.3-70b-versatile", api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")


# Dictionary to hold multiple session histories
sessions = {}
current_session = "session_1"

# Create initial session with system message
sessions[current_session] = ChatMessageHistory()
sessions[current_session].add_message(SystemMessage(content='You are a helpful AI assistant who answers in a brief fashion'))

print("💬 Session-based Chat — Manage multiple conversations")
print("Commands:")
print("  - 'switch <name>' → change or create new session")
print("  - 'list'          → list all sessions")
print("  - 'exit'          → quit the program\n")

while True:
    user_input = input(f'[{current_session}] You: ').strip()
    
    if user_input.lower() == 'exit':
        print("👋 Goodbye!")
        break
    
    # --- Switch session ---
    if user_input.startswith('switch '):
        new_session = user_input.split(' ', 1)[1]
        current_session = new_session
        
        # Create new session if not exists
        if current_session not in sessions:
            sessions[current_session] = ChatMessageHistory()
            sessions[current_session].add_message(SystemMessage(content='You are a helpful AI assistant who answers in a brief fashion'))
            print(f"✅ Created new session: {current_session}\n")
        else:
            print(f"✅ Switched to session: {current_session}\n")
        continue
    
    # --- List sessions ---
    if user_input == 'list':
        print("\n--- 🧠 All Sessions ---")
        for session_name, memory in sessions.items():
            message_count = len(memory.messages) - 1  # Exclude system message
            exchanges = message_count // 2
            marker = "👉" if session_name == current_session else "  "
            print(f"{marker} {session_name}: {exchanges} exchanges")
        print()
        continue
    
    # --- Regular conversation ---
    memory = sessions[current_session]
    
    # Add user message
    memory.add_user_message(user_input)
    
    # Get model response
    result = llama_model.invoke(memory.messages)
    
    # Add AI message
    memory.add_ai_message(result.content)
    
    print("AI:", result.content)

# --- Print all sessions at end ---
print("\n--- 📚 Final Sessions Summary ---")
for session_name, memory in sessions.items():
    print(f"\n[{session_name}]")
    for msg in memory.messages:
        role = (
            "System" if isinstance(msg, SystemMessage)
            else "User" if isinstance(msg, HumanMessage)
            else "AI"
        )
        print(f"  {role}: {msg.content}")


💬 Session-based Chat — Manage multiple conversations
Commands:
  - 'switch <name>' → change or create new session
  - 'list'          → list all sessions
  - 'exit'          → quit the program



[session_1] You:  switch s1


✅ Created new session: s1



[s1] You:  hello i am arif


AI: Hello Arif, how can I assist you today?


...

AI: Your name is Arif.


[s1] You:  who invented it


AI: Python was invented by Guido van Rossum.


[s1] You:  exit


👋 Goodbye!

--- 📚 Final Sessions Summary ---

[session_1]
  System: You are a helpful AI assistant who answers in a brief fashion

[s1]
  System: You are a helpful AI assistant who answers in a brief fashion
  User: hello i am arif
  AI: Hello Arif, how can I assist you today?
  User: what is python
  AI: Python is a high-level, easy-to-learn programming language used for web development, data analysis, and artificial intelligence.
  User: what is my name
  AI: Your name is Arif.
  User: who invented it
  AI: Python was invented by Guido van Rossum.

[s2]
  System: You are a helpful AI assistant who answers in a brief fashion
  User: hello i am rauf
  AI: Hello Rauf, how can I help you?
  User: what is my name
  AI: Your name is Rauf.
